# Finding the Critical Temperature

Running the simulation, magnetisation and energy data were provided throughout the evolution of the system. This data had to carefully analysed to determine the "shape" of the distribution of each. 

First data must be pulled from the simulations, and then can be unpacked. This is done so via the `np.load` (and previously `np.save`) function, to efficiently store data. 

The data is unpacked into dictionaries for easy access and readability. Here, for the magnetisation, we store the mean magnetisation, the susceptibility (i.e. the variance of the magnetisation) and the Binder cumulant. For energy, we store the mean, the specific heat (i.e. the variance of the energy) and the Binder cumulant. 

Currently, the "warmup" period (i.e. the period for the magnetisation/energy to stabilise, or get to a resonable state) is arbitrarily set at 1500 steps. This should be modified later to vary depending on the size and temperature of each individual system.

Below is an example for magnetisation.

In [ ]:
warmup = 1500

m = mag[warmup:]
            

total_length = len(m)
sub_sample_length = total_length//nsplits
_mean,_chi,_kurt = [],[],[]

for i in range(nsplits):
    _m = m[i*sub_sample_length:(i+1)*sub_sample_length:freq]   
    _mean.append( _m.mean()/N**2 )           
    _chi.append( _m.var()/T/N**2)

    binder = 1- np.mean(_m**4)/3./np.mean(_m**2)**2
    _kurt.append( binder) 

    mag_dict [(N,T)] = {'mag_per_particle':(np.mean(_mean),np.std(_mean)),
                        'chi_per_particle':(np.mean(_chi),np.std(_chi)),
                        'kurtosis': (np.mean(_kurt),np.std(_kurt))
                                           }

`mag_dict` stores all the information related to the magnetic field. `n_splits` determines how many times the magnetisation/energy is  "subsample", in order to estimate the error, with frequency determining how many points to select from the range. 

With this data, graphs of Binder Cumulant/Kurtosis against temperature can be plotted. As mentioned in *Theory.Binder Cumulant*, at the critical temperature, the kurtosis of each system size should be the same - *scale-invariance* is observed, due to the correlation length tending to infinity at this point. 

We can fit polynomials to the data to better see the intersection, taking care to ensure the critical temperature is near the midpoint of the temperature domain, such that the polynomial fit is accurate. 

![Fig. 1: A plot of Binder Cumulant against temperature for 6 systems of different size, with T between 2 & 2.5.](/images/mag_k-t_graph.png)

This is reasonably close to the analytical value of $T_c = 2.269\text{K}$ [ref needed]. However, the fitting is inconsistent due to the nature of polynomial fitting. This can especially be seen when using larger sets of points (see Fig. 2).

![Fig. 2: Fitting with a larger number of plots. The range of values for the intersection is difficult, and anomalous fitting artifacts cause the critical temperature to stray from the true "intersection".](/images/bad_fitting.png)

There are a few ways to potentially counteract this. One could simply limit the range of points that classify as "intersections", however this is not a general solution (this is done in Fig. 1). By lowering the degree of the polynomial fit, this could be improved, but the larger system sizes still caused issues with anomalous intersection points. 

![Fig. 3: Fitting with low degree polynomials. Large system sizes cause anomalous intersections to appear, skewing the critical temperature reading.](/images/bad_fitting_2.png)

A better approach to fitting was to use `numpy.spline` to fit a low-order polynomial between each pair of data points as shown in Fig. 4 (completeley eliminating anomalous intersections), and then use a least-variance fit to determine the point of intersection i.e. the point at which all the lines are closet to each other (Figs. 4,6).
 
![Fig. 4: Using spline fit to fit the data magnetisation data.](/images/spline_mag_good.png)

![Fig. 5: A closeup of the spline fitting for the data. Some fitting artifacts remain, however the critical point is clearly visible.](/images/spline_mag_zoomed.png)

![Fig. 6: Finding the minimal variance between spline fits for each different system size. The peak demonstrates the closest value - i.e. the intersection point. This finds a critical temperature of 2.269..., within 5 decimal places of the analytical value. ](/images/reciprocal_variance.png)

The bootstrap method can be used to estimate the error here, by running the fitting algorithm many times, with points randomly shifted according to their error. However the bootstrap method is not very useful for finding errors here, drastically reducing the actual mean value of $T_c$ that is found. The bootstrap method on this data finds a result of 

$$\text{Bootstrap with inverse variance method: } T_c = (2.256 \pm 0.057) $$

One approach to solving this might be to reduce the range of datapoints used, since larger temperatures have much larger error due to random thermal fluctuations. Limiting the range of temperatures to between 2.15-2.4K and using 25 datapoints, the value of $T_c$ found is 

$$\text{Bootstrap with inverse variance method (limited range)} T_c = (2.26878 \pm 0.00058 )$$

A plot of the fits for the reduced temperature range is shown in Fig. 7.

![Fig. 7: Spline fitting on a reduced temperature range for magnetisation. It is clear that the error towards higher temperatures is lower, simply as a result of fewer thermal fluctuations.](/images/final_reduced_range_spline_fit.png)

A better approach still might be to consider pairwise crossings, and find the mean "crossing". Continuing to use spline fits, each fit will have a point at which it crosses with every other fit. At this point, there will also be some error as a result of the error in the points creating this fit, which can then be propogated into the overall estimate of $T_c$. 

Consider first just fits, between two system sizes $L_i$ and $L_j$, with fits $U_i$ and $U_j$. Then, we can define 

$$D_{ij}(T)=U_i(T)- U_j(T)$$

implying that crossing between $U_i$ and $U_j$ occurs at $D_{ij}(T_{c, ij})=0$. This can be found via a root finding function e.g. `scipy.brentq` to find a value of $T_c$ between these two fits.

Then, to find the error in this crossing, can do a **local linear approximation**. Near the crossing $T_{c,ij}$, $D_{ij}$ can be approximated via its Taylor expansion

$$D_{ij}(T) \approx D_{ij}(T_{c,ij})+ \frac{dD_{ij}}{dT}\bigg|_{T_{c,ij}}(T-T_{c,ij})$$

Since $D_{ij}(T_{c,ij})=0$ by definition, small fluctutuations in $D_{ij}$ cause shifts (which we label $\Delta T_{c,ij}=(T-T_{c,ij})$).

Let $\Delta D_{ij}$ be the standard error on $D_{ij}$, which can be found simply by combining the errors on $U_i$ and $U_j$ 

$$\Delta D_{ij}= \sqrt{(\Delta U_i)^2+(\Delta U_j)^2}$$

This error corresponds to a "shift" in a $D_{ij}$, which in turn will cause a shift in $T$ i.e. $\Delta T$. Hence, we can rearrange the above to give

$$\Delta T_{c,ij} \approx \frac{\Delta D_{ij}}{\frac{d D_{ij}}{dT}\Big |_{T_{c,ij}}}$$

When finding the error, it is instructive to use the absolute value of the derivative in the denominator. 

Then, for each crossing, there will be a critical point $T_{c,ij}$ and its error. To combine them into a useful estimate for $T_c$, we can use **inverse-variance weighting** to estimate the mean and its error. This is more effective than a simple average and standard deviation, as here we assign a weight to each crossing

$$w_{ij}=\frac{1}{(\Delta T_{c,ij})^2}$$

such that functions with larger errors contribute less to the overall mean. The weighted average is then given by 

$$T_c = \frac{\sum_{ij}(T_{c,ij}w_{ij})}{\sum_{ij}w_{ij}}$$

with an error given by 

$$\Delta T_c = \sqrt{\frac{1}{\sum_{ij}w_{ij}}}$$

This method avoids the shortcomings of the bootstrap method outline before, since it uses local fits and simple error progression, rather than artificially adding noise to estimate the error. 

On the reduced temperature range, the crossings can be seen in Fig. 8.

![Fig. 8: A histogram of the crossing points between respective system sizes. The weighted average is shown in the legend.](/images/weighted_crossings_no_extrap.png)

This leads to a value of $T_c$ to be 

$$\text{Weighted-average pairwise method: } T_c = (2.26894 \pm 0.00015)$$

Further, a improved method might be to extrapolate the results to the true critical temperature. As metioned in the Theory section, about finite size scaling, different systems experience some "pseudocritical temperature" as a result of their finite size, which differs from the true critical temperature, which occurs for a system of infinite size. By plotting a graph of the temperature found against the average inverse system size of the pair, the trend can be extrapolated to zero to find the "true critical temperature". The results of this are shown in Fig. 9.

![Fig. 9: The critical temperature for each pair against its inverse system size. Fitted via a linear fit and extrapolated to zero gives $T_c=2.961$](/images/extrap_T_vs_inv_system.png)

This leads to a value of $T_c$ to be

$$\text{Extrapolating pairwise temperatures method: } T_c = (2.26911 \pm 0.00024)$$

It is clear that this last method is closest to the true value of $T_c$ given analytically via

$$T_c = \frac{2}{\ln{(1+\sqrt{2})}} \approx 2.26918531...$$